# 11 · Static vs continuous batching vs chunked prefill

Part of **[Inference Engineering](https://github.com/supriyo100/kv-cache-attention-variants)**, an original notebook (not from the upstream course).
Derivation: path step 9. Sources: Orca (OSDI 2022); Sarathi-Serve (arXiv:2403.02310).

Every number below is produced by code in this notebook or in `inference_lab` (tested in
`tests/`). Labels: *measured* = ran here (CPU), *theoretical* = formula with datasheet inputs.

In [ ]:
# Setup: works locally (uv sync) and on Colab (installs the small inference_lab package only)
try:
    import inference_lab  # noqa: F401
except ImportError:  # Colab / fresh environment
    %pip install -q --no-deps "git+https://github.com/supriyo100/kv-cache-attention-variants"
import torch
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (7.5, 3.6), "axes.spines.top": False, "axes.spines.right": False})
C = {"compute": "#0E7C72", "memory": "#B5790A", "waste": "#B42B3E", "shared": "#5B47C9", "ink": "#1C232B"}

## Objective
Compare scheduling policies on the same Poisson trace: throughput, TTFT, TPOT, p99 ITL and
preemptions.

## Theory and equations
$\mathcal{R}_{t+1} = (\mathcal{R}_t \setminus \mathcal{F}_t) \cup \mathcal{A}_t$. Chunked prefill caps
the tokens per iteration at $\tau$, so iteration time, and therefore ITL, is bounded. Each forward
pass is timed with the roofline model: $t = \max(F/\pi, Q/\beta)$.

## Diagram
```
static:      [prefill A..F][decode ......... until the longest finishes][prefill G..L] ...
continuous:  A B C D E F | F done -> G admitted next iteration | ...
chunked:     every iteration = all decodes + up to (tau - decodes) prompt tokens
```

## Implementation

In [ ]:
from inference_lab import scheduler as S
trace = S.poisson_trace(200, rate=20.0, seed=3)

## Experiment (*theoretical*: roofline-timed, Llama-3-8B on one H100)

In [ ]:
rows = [S.simulate(trace, p) for p in ("static", "continuous", "chunked")]
print(f"{'policy':11s} {'tok/s':>7s} {'TTFT':>8s} {'TTFT p99':>9s} {'TPOT':>8s} {'ITL p99':>8s} {'preempt':>8s}")
for r in rows:
    print(f"{r.policy:11s} {r.throughput_tok_s:7.0f} {r.ttft_mean:7.3f}s {r.ttft_p99:8.3f}s "
          f"{1e3*r.tpot_mean:6.1f}ms {1e3*r.itl_p99:6.1f}ms {r.preemptions:8d}")

## Results: load sweep

In [ ]:
rates = [5, 10, 15, 20, 25, 30]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
for p, col in [("static", C["waste"]), ("continuous", C["memory"]), ("chunked", C["compute"])]:
    rs = [S.simulate(S.poisson_trace(150, rate=r, seed=1), p) for r in rates]
    ax[0].plot(rates, [x.ttft_mean for x in rs], "o-", color=col, label=p)
    ax[1].plot(rates, [1e3 * x.itl_p99 for x in rs], "o-", color=col, label=p)
ax[0].set_xlabel("arrival rate (req/s)"); ax[0].set_ylabel("mean TTFT (s)"); ax[0].set_yscale("log")
ax[1].set_xlabel("arrival rate (req/s)"); ax[1].set_ylabel("p99 ITL (ms)"); ax[1].legend()
plt.tight_layout(); plt.show()

## Interpretation
Static batching has smooth decode but queues requests behind whole batches. Prefill-first
continuous batching fixes TTFT and throughput, and creates ITL spikes. Chunked prefill keeps
the TTFT gains and bounds ITL.

## Limitations
Roofline timing is an upper bound on speed. No CPU scheduling overhead, no CUDA graphs, no
prefix caching. Static mode reserves each request's true length, which is generous to it.

## Conclusion
The scheduling policy moves latency percentiles by an order of magnitude on identical hardware.